# EDA + Limpieza básica del dataset

Objetivo:
- Entender el dataset
- Analizar la calidad del texto
- Detectar problemas (nulos, ruido, longitud, encoding residual)
- Aplicar limpieza ligera SIN perder información

⚠️ Nota:
No se eliminan duplicados en esta etapa.

In [ ]:
import pandas as pd
import re

In [ ]:
df = pd.read_csv("../data/processed/data_fixed_v2md.csv")
df.head()

In [ ]:
# Información general

print("Shape:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

In [ ]:
df.isnull().sum()

In [ ]:
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
text_cols

In [ ]:
df["tweet_text_len"] = df["tweet_text"].astype(str).str.len()
df["tweet_text_len"].describe()

In [ ]:
df[df["tweet_text_len"] < 10]["tweet_text"].head(20)

In [ ]:
df["tweet_text"].sample(20, random_state=42)

In [ ]:
bad_pattern = re.compile(r"[ÃÂâ]")

def has_suspicious_text(x):
    return isinstance(x, str) and bool(bad_pattern.search(x))

df["tweet_text"].apply(has_suspicious_text).sum()

In [ ]:
df[df["tweet_text"].apply(has_suspicious_text)]["tweet_text"].head(10)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return text
    
    text = text.lower()
    
    # quitar URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # quitar menciones (@usuario)
    text = re.sub(r"@\w+", "", text)
    
    # normalizar espacios
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [ ]:
df["tweet_text_clean"] = df["tweet_text"].apply(clean_text)

In [ ]:
comparison = pd.DataFrame({
    "before": df["tweet_text"].head(10),
    "after": df["tweet_text_clean"].head(10)
})

comparison

In [ ]:
df["tweet_text_clean"].sample(20, random_state=42)